<a href="https://colab.research.google.com/github/hafsabaloch281-creator/Before_Alignment_Model/blob/main/Before_Alignment_Base_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# Install correct package versions to resolve dependency conflicts and warnings
!pip install -q -U trl transformers datasets accelerate matplotlib
!pip install -q "pandas==2.2.3" "bottleneck>=1.5.0"

In [5]:
import torch, pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer

# Detect if GPU is available
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
print(f"Using device: {device} with dtype: {dtype}")

name = "Qwen/Qwen2.5-0.5B"
tok = AutoTokenizer.from_pretrained(name)
base = AutoModelForCausalLM.from_pretrained(name, torch_dtype=dtype).to(device)

prompts = [
 "Explain what In-Contextual Learning.",
 "What is the difference between Direct Preference Optimization and Supervised Fine Tubing?",
 "I have an assignment to complete within a week on RLHF. What should I do, which model should I select and which dataset should I select for this topic?",
 "Write a brief description about weather conditon of Gwadar.",
 "A shop sells 4 cham for 25 rupees. How much will 15 cham cost? Explain step by step.",
 "What is your favourite subject and why was it chosen as the favourite subject?",
]

def generate(model, prompt):
    text = tok.apply_chat_template([{"role":"user","content":prompt}],
                                   tokenize=False, add_generation_prompt=True)
    inp = tok(text, return_tensors="pt").to(device)
    out = model.generate(**inp, max_new_tokens=200, do_sample=False,
                         pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][inp.input_ids.shape[1]:], skip_special_tokens=True)

base_out = {p: generate(base, p) for p in prompts}
for p, r in base_out.items(): print("PROMPT:", p, "\nBASE:", r, "\n", "-"*60)

Using device: cpu with dtype: torch.float32


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

PROMPT: Explain what In-Contextual Learning. 
BASE: In-Contextual Learning (ICL) is a learning approach that focuses on the context in which a learner is learning. It emphasizes the importance of understanding the context in which a learner is learning, as well as the impact of that context on their learning outcomes. ICL is based on the idea that learning is not just about acquiring knowledge, but also about understanding the context in which that knowledge is being applied. By considering the context in which a learner is learning, ICL helps learners to develop a deeper understanding of the subject matter and to apply that knowledge in a meaningful way. ICL is often used in educational settings to help students develop critical thinking skills, problem-solving abilities, and a deeper understanding of the world around them. 
 ------------------------------------------------------------
PROMPT: What is the difference between Direct Preference Optimization and Supervised Fine Tubing? 
B

In [7]:
import sys
import torch
from transformers import TextIteratorStreamer
from threading import Thread

def generate_interactive(model, tokenizer):
    user_prompt = input("Enter your prompt: ")

    # Apply chat template
    formatted_text = tokenizer.apply_chat_template(
        [{"role": "user", "content": user_prompt}],
        tokenize=False,
        add_generation_prompt=True
    )

    # Tokenize (using the dynamically detected 'device')
    inputs = tokenizer([formatted_text], return_tensors="pt").to(device)

    # Setup a streamer for real-time token-by-token output
    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

    generation_kwargs = dict(
        inputs,
        streamer=streamer,
        max_new_tokens=200,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    # Run generation in a separate thread so we can stream tokens in the main thread
    thread = Thread(target=model.generate, kwargs=generation_kwargs)
    thread.start()

    print("\n--- Generating Output Token-by-Token ---")
    for new_text in streamer:
        print(new_text, end="", flush=True)
    print("\n----------------------------------------")

# Run the interactive generation function
generate_interactive(base, tok)

Enter your prompt: "A project team consists of four members: Ali, Bilal, Sana, and Zara.  Ali must review code before Sana can deploy it.  Zara cannot review code until Bilal has completed his unit tests.  Bilal finishes his unit tests after Sana deploys the code. Based on these rules, what is the exact chronological sequence of the team members' tasks? Explain your reasoning step by step."

--- Generating Output Token-by-Token ---
To determine the exact chronological sequence of the team members' tasks, we need to follow the given rules and their corresponding actions:

1. **Ali must review code before Sana can deploy it.**
   - This means that Sana must first review the code before it can be deployed.

2. **Zara cannot review code until Bilal has completed his unit tests.**
   - This means that Zara must wait until Bilal has finished his unit tests before she can review the code.

3. **Bilal finishes his unit tests after Sana deploys the code.**
   - This means that Bilal must finish